# 06 FTMO Vs Standard

Notebook n?y so s?nh c?ng m?t chi?n l??c Combo d??i 2 ch? ?? qu?n tr? v?n:
- `standard`
- `ftmo`

B?n c? th? ch?y ? m?c **symbol** v? m?c **portfolio** trong c?ng m?t notebook.


In [ ]:
# Bootstrap: add repo root + core_python to sys.path
import sys
from pathlib import Path

def _find_root(start: Path, marker: str = 'config.py') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f'Could not locate repo root containing {marker!r}')

ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('dark_background')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

from strategies.combo.config import summary as strategy_summary
from strategies.combo.portfolio.backtest import compare_account_modes
from strategies.combo.symbol.backtest import run_symbol_backtest

print(strategy_summary())


In [ ]:
SYMBOL = 'US30'
PORTFOLIO_SYMBOLS = ['US30', 'US500', 'DE40', 'GOLD']
INITIAL_BALANCE = 100_000.0
DATE_FROM = '2023-01-01'
DATE_TO = None
MAX_BARS = 30000


In [ ]:
symbol_results = {}
for mode in ['standard', 'ftmo']:
    symbol_results[mode] = run_symbol_backtest(
        SYMBOL,
        init_eq=INITIAL_BALANCE,
        account_mode=mode,
        date_from=DATE_FROM,
        date_to=DATE_TO,
        max_bars=MAX_BARS,
    )

symbol_compare = pd.DataFrame({
    mode: {k: v for k, v in res.metrics.items() if k != 'monthly_pnl_table'}
    for mode, res in symbol_results.items()
}).T
display(symbol_compare)


In [ ]:
mode_results = compare_account_modes(
    symbol_keys=PORTFOLIO_SYMBOLS,
    initial_balance=INITIAL_BALANCE,
    date_from=DATE_FROM,
    date_to=DATE_TO,
    max_bars=MAX_BARS,
)

portfolio_compare = pd.DataFrame({
    mode: {k: v for k, v in res.metrics.items() if k != 'monthly_pnl_table'}
    for mode, res in mode_results.items()
}).T
display(portfolio_compare)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)

for mode, res in symbol_results.items():
    res.equity.plot(ax=axes[0], lw=1.8, label=f'{SYMBOL} - {mode}')
axes[0].set_title('Symbol equity: standard vs ftmo')
axes[0].grid(alpha=0.3)
axes[0].legend()

for mode, res in mode_results.items():
    res.combined_equity.plot(ax=axes[1], lw=1.8, label=f'portfolio - {mode}')
axes[1].set_title('Portfolio equity: standard vs ftmo')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
